# From a Harmonic Oscillator to a Qubit

A simple LC circuit behaves as a quantum harmonic oscillator with equally spaced energy levels.

Such a system cannot be used as a qubit because a microwave drive resonant with one transition is automatically resonant with all transitions.

The transmon solves this problem by replacing the linear inductor with a Josephson junction. The Josephson energy

$$
-E_J \cos \varphi
$$

introduces a weak nonlinearity, making the energy levels slightly unequally spaced.

Use the sliders to explore how the transmon spectrum depends on:

- the Josephson energy $E_J$,
- the charging energy $E_C$,
- the ratio $E_J/E_C$.

Questions:

1. What happens when $E_J/E_C$ becomes very large?
2. How does the anharmonicity evolve?
3. Why is a transmon often described as an "almost harmonic oscillator"?

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from ipywidgets import interact, FloatSlider

In [2]:
def transmon_levels(EJ=15, EC=0.3):

    N = 25

    n = np.arange(-N,N+1)

    H = np.diag(4*EC*n**2)

    for i in range(len(n)-1):
        H[i,i+1] += -EJ/2
        H[i+1,i] += -EJ/2

    E,_ = eigh(H)

    return E[:6]

In [3]:
def plot_transmon(EJ=15, EC=0.3):

    E = transmon_levels(EJ,EC)

    phi = np.linspace(-np.pi,np.pi,500)

    V = -EJ*np.cos(phi)

    plt.figure(figsize=(7,5))

    plt.plot(phi,V)

    for e in E:
        plt.axhline(e)

    plt.xlabel(r"$\varphi$")
    plt.ylabel("Energy")

    plt.title(
        rf"$E_J/E_C={EJ/EC:.1f}$"
    )

    plt.show()

    print(
        "Anharmonicity ≈",
        (E[2]-E[1])-(E[1]-E[0])
    )

interact(
    plot_transmon,
    EJ=FloatSlider(
        value=15,
        min=1,
        max=50,
        step=1
    ),
    EC=FloatSlider(
        value=0.3,
        min=0.05,
        max=2,
        step=0.05
    )
);

interactive(children=(FloatSlider(value=15.0, description='EJ', max=50.0, min=1.0, step=1.0), FloatSlider(valu…

# Why the Transmon Works: Charge Dispersion

The transmon Hamiltonian is

$$
H = 4E_C(n-n_g)^2 - E_J\cos\varphi .
$$

Here $n_g$ is an offset charge induced by the electrostatic environment.

If the qubit transition frequency depends strongly on $n_g$, then small charge fluctuations will dephase the qubit.

Use the slider $E_J/E_C$ to observe the key idea of the transmon:

- increasing $E_J/E_C$ makes the spectrum more harmonic,
- but it also makes the energy levels almost flat as a function of $n_g$,
- therefore the qubit becomes much less sensitive to charge noise.

This is the central compromise of the transmon: sacrificing some anharmonicity to gain much longer coherence.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from ipywidgets import interact, FloatSlider, IntSlider

def transmon_spectrum_vs_ng(EJ_over_EC=50, EC=1.0, n_levels=5):
    EJ = EJ_over_EC * EC

    # Charge basis |n>, with n Cooper pairs
    N = 35
    n = np.arange(-N, N + 1)

    ng_values = np.linspace(-1.0, 1.0, 301)
    levels = []

    for ng in ng_values:
        # Hamiltonian: 4 EC (n-ng)^2 - EJ cos(phi)
        # In charge basis, cos(phi) couples n to n±1 with matrix element 1/2.
        H = np.diag(4 * EC * (n - ng)**2)

        for i in range(len(n) - 1):
            H[i, i + 1] += -EJ / 2
            H[i + 1, i] += -EJ / 2

        E, _ = eigh(H)
        levels.append(E[:n_levels])

    levels = np.array(levels)

    # Shift all levels by E0 at ng=0 for readability
    idx0 = np.argmin(np.abs(ng_values))
    levels_shifted = levels - levels[idx0, 0]

    plt.figure(figsize=(8, 5))

    for k in range(n_levels):
        plt.plot(
            ng_values,
            levels_shifted[:, k],
            label=rf"$E_{k}$"
        )

    plt.xlabel(r"Offset charge $n_g$")
    plt.ylabel(r"Energy relative to $E_0(n_g=0)$")
    plt.title(rf"Transmon spectrum vs offset charge, $E_J/E_C={EJ_over_EC:.1f}$")
    plt.grid(True)
    plt.legend()
    plt.show()

    # Qubit transition frequency and charge dispersion
    f01 = levels[:, 1] - levels[:, 0]
    charge_dispersion = np.max(f01) - np.min(f01)

    anharmonicity = (
        (levels[idx0, 2] - levels[idx0, 1])
        -
        (levels[idx0, 1] - levels[idx0, 0])
    )

    print(f"Charge dispersion of f01: {charge_dispersion:.4e} EC")
    print(f"Anharmonicity at ng=0: {anharmonicity:.4f} EC")

def plot_transmon_charge_dispersion():
    interact(
        transmon_spectrum_vs_ng,
        EJ_over_EC=FloatSlider(
            value=50,
            min=1,
            max=100,
            step=1,
            description=r"$E_J/E_C$"
        ),
        EC=FloatSlider(
            value=1.0,
            min=0.2,
            max=2.0,
            step=0.1,
            description=r"$E_C$"
        ),
        n_levels=IntSlider(
            value=5,
            min=2,
            max=8,
            step=1,
            description="levels"
        )
    )

plot_transmon_charge_dispersion()

interactive(children=(FloatSlider(value=50.0, description='$E_J/E_C$', min=1.0, step=1.0), FloatSlider(value=1…